In [0]:
%run ../../config/utils

In [0]:
import pandas as pd
import mlflow
import sys

mlflow.set_registry_uri('databricks-uc')

In [0]:
recent_saturday = pd.Timestamp.today() - pd.offsets.Week(weekday=5)
recent_saturday_str = recent_saturday.strftime('%Y-%m-%d')
trips_model_uri = f"models:/{digital_trips_catalog}@champion"

In [0]:
trips_model = mlflow.xgboost.load_model(trips_model_uri)
base_Data_v2 = spark.table(digital_propensity_features).filter(f.col('FISCAL_WEEK_END') == recent_saturday_str)
try:
    base_Data_pd = base_Data_v2.toPandas()
    print("Data converted to Pandas")
except Exception as e:
    print("Error converting to Pandas:", e)
    sys.exit(1)

In [0]:

base_Data_pd["trips_score"] = trips_model.predict_proba(base_Data_pd[['AVERAGE_DAYS_BETWEEN_LAST_180DAYS',
 'BEFORE_LAST_Q60DAYS_DAYTRIPS',
 'AVERAGE_DAYS_BETWEEN_LAST_60DAYS',
 'LAST_60DAYS_PERISHABLES_TRIPS',
 'LAST_60DAYS_GROCERY_TRIPS',
 'LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS',]])[:, 1]


base_Data_pd_id = base_Data_pd[['MBRSHP_SID','FISCAL_WEEK_END','LATEST_MBRSHP_NBR','trips_score']]

In [0]:
df = spark.createDataFrame(base_Data_pd_id)
df.write.mode('overwrite').option('replaceWhere', f"FISCAL_WEEK_END = '{recent_saturday_str}'").saveAsTable(digital_trips_scores)

In [0]:
spark.sql(f"""
    DELETE FROM {digital_trips_scores} WHERE FISCAL_WEEK_END < '{(recent_saturday - pd.offsets.Day(365)).date()}'
""")

In [0]:
dbutils.library.restartPython()